# Task 2.4
Before training, verify your analytical gradients from Task 2.3 against numerical finite differences. For a randomly initialised network, compute both the analytical gradient $\frac{\partial L}{\partial W_{ij}^{(1)}}$ and the finite difference approximation:

$$
\frac{\partial L}{\partial W_{ij}^{(1)}} \approx
\frac{
L\left(W_{ij}^{(1)}+\epsilon\right)
-L\left(W_{ij}^{(1)}-\epsilon\right)
}{
2\epsilon
}
\tag{8}
$$

using $\epsilon=10^{-5}$

In [26]:
%%capture
%run Task2.ipynb

In [2]:
 # imports
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import math

## Equation
The two equations I have derived and will compare are:

**Gradient with respect to $W_n^{(1)}$:**

$$
\begin{aligned}
\frac{\partial L}{\partial W_n^{(1)}}
={}&
\frac{2}{N_d}\sum_{i=1}^{N_d}
\left(NN(t_i)-x(t_i)\right)
W_n^{(2)}t_i
\left(1-\tanh^2(z_{n,i})\right)
\\[6pt]
&+\lambda_{phys}\frac{2}{N_c}
\sum_{i=1}^{N_c}r_i
\Bigg[
-4W_n^{(2)}W_n^{(1)}
\tanh(z_{n,i})
\left(1-\tanh^2(z_{n,i})\right)
\\
&\quad
-2W_n^{(2)}(W_n^{(1)})^2t_i
\left(1-\tanh^2(z_{n,i})\right)^2
\\
&\quad
+4W_n^{(2)}(W_n^{(1)})^2t_i
\tanh^2(z_{n,i})
\left(1-\tanh^2(z_{n,i})\right)
\\
&\quad
+\frac{\mu}{m}
\Bigg(
W_n^{(2)}
\left(1-\tanh^2(z_{n,i})\right)
\\
&\qquad
-2W_n^{(2)}W_n^{(1)}t_i
\tanh(z_{n,i})
\left(1-\tanh^2(z_{n,i})\right)
\Bigg)
\\
&\quad
+\frac{k}{m}W_n^{(2)}t_i
\left(1-\tanh^2(z_{n,i})\right)
\Bigg]
\\[6pt]
&+\lambda_{IC}
\Bigg[
2\left(
\sum_{j=1}^{H}W_j^{(2)}W_j^{(1)}
\left(1-\tanh^2(b_j^{(1)})\right)
\right)
W_n^{(2)}
\left(1-\tanh^2(b_n^{(1)})\right)
\Bigg]
\end{aligned}
$$

**Gradient with respect to $W_n^{(2)}$:**

$$
\begin{aligned}
\frac{\partial L}{\partial W_n^{(2)}}
={}&
\frac{2}{N_d}\sum_{i=1}^{N_d}
\left(NN(t_i)-x(t_i)\right)
\tanh(z_{n,i})
\\[6pt]
&+\lambda_{phys}\frac{2}{N_c}
\sum_{i=1}^{N_c}r_i
\Bigg[
-2(W_n^{(1)})^2
\tanh(z_{n,i})
\left(1-\tanh^2(z_{n,i})\right)
\\
&\quad
+\frac{\mu}{m}W_n^{(1)}
\left(1-\tanh^2(z_{n,i})\right)
\\
&\quad
+\frac{k}{m}\tanh(z_{n,i})
\Bigg]
\\[6pt]
&+\lambda_{IC}
\Bigg[
2\left(
\sum_{j=1}^{H}W_j^{(2)}
\tanh(b_j^{(1)})+b^{(2)}-1
\right)
\tanh(b_n^{(1)})
\\
&\quad
+2\left(
\sum_{j=1}^{H}W_j^{(2)}W_j^{(1)}
\left(1-\tanh^2(b_j^{(1)})\right)
\right)
W_n^{(1)}
\left(1-\tanh^2(b_n^{(1)})\right)
\Bigg]
\end{aligned}
$$

**Loss function:**
$$
L = L_{data} + \lambda_{phys}\cdot L_{phys} + \lambda_{IC}\cdot L_{IC}
$$
**Loss function expanded:**

$$
\begin{aligned}
L ={}&
\frac{1}{N_d}\sum_{i=1}^{N_d}
\left(
NN(t_i)-e^{-2.5t_i}
\left(2\cdot0.5\cdot\cos(-0.11+22.22t_i)\right)
\right)^2
\\[6pt]
&+\lambda_{phys}\frac{1}{N_c}\sum_{i=1}^{N_c}
\left(
\frac{d^2NN}{dt^2}(t_i)
+\frac{\mu}{m}\frac{dNN}{dt}(t_i)
+\frac{k}{m}NN(t_i)
\right)^2
\\[6pt]
&+\lambda_{IC}
\left[
\left(NN(0)-1\right)^2
+\left(\frac{dNN}{dt}(0)\right)^2
\right]
\end{aligned}
$$

**Neural Network**
$$
NN(t_i) = \sum_{n=1}^{H} W_n^{(2)} \tanh\left(W_n^{(1)}t_i + b_n^{(1)}\right) + b^{(2)}
$$

**First derivative of the neural network:**

$$
\frac{dNN}{dt}(t_i) =
\sum_{n=1}^{H}
W_n^{(2)}W_n^{(1)}
\left(
1-\tanh^2\left(W_n^{(1)}t_i+b_n^{(1)}\right)
\right)
$$

**Second derivative of the neural network:**

$$
\frac{d^2NN}{dt^2}(t_i) =
-2\sum_{n=1}^{H}
W_n^{(2)}(W_n^{(1)})^2
\tanh\left(W_n^{(1)}t_i+b_n^{(1)}\right)
\left(
1-\tanh^2\left(W_n^{(1)}t_i+b_n^{(1)}\right)
\right)
$$

**Loss function with the neural network and its derivatives fully substituted:**

$$
\begin{aligned}
L ={}&
\frac{1}{N_d}\sum_{i=1}^{N_d}
\Bigg(
\sum_{n=1}^{H}
W_n^{(2)}
\tanh\left(W_n^{(1)}t_i+b_n^{(1)}\right)
+b^{(2)}
-e^{-2.5t_i}
\left(2\cdot0.5\cdot\cos(-0.11+22.22t_i)\right)
\Bigg)^2
\\[10pt]
&+\lambda_{phys}\frac{1}{N_c}
\sum_{i=1}^{N_c}
\Bigg[
-2\sum_{n=1}^{H}
W_n^{(2)}(W_n^{(1)})^2
\tanh\left(W_n^{(1)}t_i+b_n^{(1)}\right)
\left(
1-\tanh^2\left(W_n^{(1)}t_i+b_n^{(1)}\right)
\right)
\\[6pt]
&\qquad
+\frac{\mu}{m}
\sum_{n=1}^{H}
W_n^{(2)}W_n^{(1)}
\left(
1-\tanh^2\left(W_n^{(1)}t_i+b_n^{(1)}\right)
\right)
\\[6pt]
&\qquad
+\frac{k}{m}
\left(
\sum_{n=1}^{H}
W_n^{(2)}
\tanh\left(W_n^{(1)}t_i+b_n^{(1)}\right)
+b^{(2)}
\right)
\Bigg]^2
\\[10pt]
&+\lambda_{IC}
\Bigg[
\left(
\sum_{n=1}^{H}
W_n^{(2)}\tanh\left(b_n^{(1)}\right)
+b^{(2)}-1
\right)^2
\\[6pt]
&\qquad
+\left(
\sum_{n=1}^{H}
W_n^{(2)}W_n^{(1)}
\left(
1-\tanh^2\left(b_n^{(1)}\right)
\right)
\right)^2
\Bigg]
\end{aligned}
$$

Firstly, creating the neural network

In [29]:
#constants
m = 1.0 # mass
mu = 5 # friction constant
k = 500 # spring constant

omega_intial = np.sqrt(k)
delta = mu/(2*m)
omega = np.sqrt(omega_intial**2 - delta**2)

def A(phi):
    return 1 / (2*math.cos(phi))

phi = np.arctan(-1*delta/omega)

t_train = torch.linspace(start = 0, end = 0.5, steps = 10)
t_train = t_train.view(-1,1) # We transform it into shape (samples, features), instead of (samples). In our case that is (10,1)

def x(t):
    return  np.exp(-1*delta*t) *(2*A(phi)* np.cos(omega*t + phi))
y_train = x(t_train)

C:\Users\Saads\AppData\Local\Temp\ipykernel_24308\4019892306.py:19: DeprecationWarning:

__array_wrap__ must accept context and return_scalar arguments (positionally) in the future. (Deprecated NumPy 2.0)



In [30]:
t_train = t_train.reshape(-1, 1)
t_train = t_train.detach().clone().requires_grad_(True).double()
t0 = torch.tensor([0.0], requires_grad=True)
# Creating the model
model = nn.Sequential(nn.Linear(1, 10), nn.Tanh(), nn.Linear(10, 1))
model = model.double()

# Making NN a function of t and then deriving it
# def NN(t):
#     return model(t)
# NN = model(t_train)
# dNN_dt = torch.autograd.grad(NN, t_train, grad_outputs=torch.ones_like(NN), create_graph=True)[0]
# dNN2_dt2 = torch.autograd.grad(dNN_dt, t_train, grad_outputs=torch.ones_like(dNN_dt), create_graph=True)[0]

# # Physics loss. The residuals must equal 0
# rt = dNN2_dt2 + (mu/m)*dNN_dt + (k/m)*NN
# loss_phys = torch.mean(rt**2)

# # Initial conditions loss. x(0) = 1 and x'(0) = 0
# NN_0 = model(t0)
# dNN_0 = torch.autograd.grad(NN_0, t0, create_graph=True)[0]
# loss_IC1 = torch.mean((NN_0 - 1)**2)
# loss_IC2 = torch.mean(dNN_0**2)

# # Data loss. The model must provide the same output as the closed form solution
# # Version 1
# # loss_data = []
# # for i in range(len(t_train)):
# #     data_loss_increment = torch.mean((model(t_train[i]) - y_train[i])**2)
# #     loss_data.append(data_loss_increment)
# # loss_data = torch.stack(loss_data).mean()
# # Version 2
# predictions = model(t_train)
# loss_fn = nn.MSELoss()
# assert predictions.shape == y_train.shape
# loss_data = loss_fn(predictions, y_train)


In [31]:
def calc_loss():
    t_col = torch.linspace(start = 0, end = 1, steps = 50).reshape(-1, 1).detach().clone().requires_grad_(True).double()
    NN = model(t_col)
    dNN_dt = torch.autograd.grad(NN, t_col, grad_outputs=torch.ones_like(NN), create_graph=True)[0]
    dNN2_dt2 = torch.autograd.grad(dNN_dt, t_col, grad_outputs=torch.ones_like(dNN_dt), create_graph=True)[0]

    # Physics loss. The residuals must equal 0
    rt = dNN2_dt2 + (mu/m)*dNN_dt + (k/m)*NN
    loss_phys = torch.mean(rt**2)

    # Initial conditions loss. x(0) = 1 and x'(0) = 0
    t0 = torch.tensor([0.0], requires_grad=True).double()
    NN_0 = model(t0)
    dNN_0 = torch.autograd.grad(NN_0, t0, create_graph=True)[0]
    loss_IC1 = torch.mean((NN_0 - 1)**2)
    loss_IC2 = torch.mean(dNN_0**2)

    # Data loss. The model must provide the same output as the closed form solution
    # Version 1
    # loss_data = []
    # for i in range(len(t_train)):
    #     data_loss_increment = torch.mean((model(t_train[i]) - y_train[i])**2)
    #     loss_data.append(data_loss_increment)
    # loss_data = torch.stack(loss_data).mean()
    # Version 2
    predictions = model(t_train)
    loss_fn = nn.MSELoss()
    
    assert predictions.shape == y_train.shape
    loss_data = loss_fn(predictions, y_train)
    
    lam_phys = 1e-4
    lam_ic = 1

    total_loss = loss_data + loss_phys * lam_phys + (loss_IC1 + loss_IC2) * lam_ic

    return total_loss


In [32]:
# print("Total loss shape: ", loss.shape)
# print("Total loss: ", loss.item())
# for i in range(10):
#     print(f'w({i}):', model[0].weight[i,0].item())
# Finite difference approximation

# w1_plus = model[0].weight.tolist()
# w1_minus = model[0].weight.tolist()
# b1 = model[0].bias.tolist()
# w2 = model[2].weight[0].tolist()
# b2 = model[2].bias.item()
e = 1e-5
fda_list_w1 = []
fda_list_w2 = []
l_plus_w1 = []
l_minus_w1 = []
l_plus_w2 = []
l_minus_w2 = []
loss_list_plus = []
loss_list_minus = []

for i in range(10):
    og_weight = model[0].weight[i,0].item()
    with torch.no_grad():
        model[0].weight[i, 0] = og_weight + e
    l_plus_w1.append(calc_loss().item())

    with torch.no_grad():
        model[0].weight[i, 0] = og_weight

    with torch.no_grad():
        model[0].weight[i, 0] = og_weight - e
    l_minus_w1.append(calc_loss().item())

    with torch.no_grad():
        model[0].weight[i, 0] = og_weight

    fda_list_w1.append((l_plus_w1[-1] - l_minus_w1[-1]) / (2 * e))

for i in range(10):
    og_weight = model[2].weight[0, i].item()
    with torch.no_grad():
        model[2].weight[0, i] = og_weight + e
    l_plus_w2.append(calc_loss().item())

    with torch.no_grad():
        model[2].weight[0, i] = og_weight

    with torch.no_grad():
        model[2].weight[0, i] = og_weight - e
    l_minus_w2.append(calc_loss().item())

    with torch.no_grad():
        model[2].weight[0, i] = og_weight

    fda_list_w2.append((l_plus_w2[-1] - l_minus_w2[-1]) / (2 * e))

print("W1 Plus losses:", l_plus_w1)
print("W1 Minus losses:", l_minus_w1)
print("W2 Plus losses:", l_plus_w2)
print("W2 Minus losses:", l_minus_w2)
print("W1 Numerical gradient: ", fda_list_w1)
print("W2 Numerical gradient: ", fda_list_w2)

W1 Plus losses: [8.474444300090967, 8.474433649329397, 8.474455015845091, 8.474449430734298, 8.474464194184497, 8.47447806136088, 8.474452538899145, 8.474444503314311, 8.474466362075004, 8.474463041638463]
W1 Minus losses: [8.474472191873767, 8.474482842544688, 8.474461476106, 8.474467061206713, 8.474452297841585, 8.474438430497178, 8.474463953025841, 8.474471988556962, 8.47445012998692, 8.47445345039188]
W2 Plus losses: [8.47456774946724, 8.474616705527533, 8.4746354748866, 8.474593299007928, 8.47457970074992, 8.474291069681083, 8.474688451377503, 8.474653693490845, 8.47464203672589, 8.474673269628159]
W2 Minus losses: [8.474348743449571, 8.474299788159744, 8.474281019456772, 8.474323194220439, 8.474336792516203, 8.474625424480106, 8.474228044368537, 8.474262801151996, 8.474274457840936, 8.474243225636942]
W1 Numerical gradient:  [-1.3945891399913533, -2.4596607645754887, -0.3230130454667801, -0.8815236207659892, 0.5948171455827378, 1.9815431850922491, -0.5707063348125985, -1.37426213

In [33]:
model.zero_grad()

total_loss = calc_loss()
total_loss.backward()

dW1 = model[0].weight.grad[:, 0].tolist()  # 10 first-layer gradients
dW2 = model[2].weight.grad[0, :].tolist()  # 10 second-layer gradients
print("dw1: ", dW1)
print("dw2: ", dW2)

dw1:  [-1.3945891399051873, -2.4596607646920723, -0.32301304551658017, -0.8815236207859675, 0.5948171456571062, 1.9815431851006031, -0.5707063347917761, -1.3742621325335829, 0.8116044041750627, 0.4795623290697304]
dw2:  [10.950300883397528, 15.845868389442568, 17.722771491350493, 13.505239374481695, 12.145411685889417, -16.717739951191277, 23.02035044819757, 19.544616942452826, 18.378944247696463, 21.502199560876452]


In [34]:
dW1 = np.asarray(dW1)
dW2 = np.asarray(dW2)
fda_list_w1 = np.asarray(fda_list_w1)
fda_list_w2 = np.asarray(fda_list_w2)

absolute_error_w1 = np.abs(dW1 - fda_list_w1)
absolute_error_w2 = np.abs(dW2 - fda_list_w2)
fig = px.line(y = [absolute_error_w1, absolute_error_w2], title='Absolute gradient error')
fig.update_layout(xaxis_title='Weight index', yaxis_title='Absolute error', template='plotly_dark', 
                  width=800, height=500, showlegend=True)
fig.show()

# 2.5

In [36]:
epoch = 20000
optimizer = torch.optim.Adam(model.parameters(), lr = 0.01)
losses = []

for i in range(epoch):
  # Loss function  where the forward pass also happens
  W1 = model[0].weight.detach().numpy()[:, 0]
  b1 = model[0].bias.detach().numpy()
  W2 = model[2].weight.detach().numpy()[0, :]
  b2 = model[2].bias.detach().numpy()[0]

  losses.append(loss(W1, b1, W2, b2))

  # Update weights
  dW1, db1, dW2, db2 = gradients(W1, b1, W2, b2)
  model[0].weight.grad = torch.tensor(dW1).reshape(10, 1)
  model[0].bias.grad = torch.tensor(db1)
  model[2].weight.grad = torch.tensor(dW2).reshape(1, 10)
  model[2].bias.grad = torch.tensor([db2], dtype=torch.float64)

  optimizer.step()

  if i % 1000 == 0:
      print(i, losses[-1])

0 8.474458239594055
1000 1.0952572569474084
2000 0.6365674401367831
3000 0.25618949727691154
4000 0.22086046684752364
5000 0.16830074431235192
6000 0.10456179366342938
7000 0.06777413570114325
8000 0.05770349098601551
9000 0.054112262685189184
10000 0.05102329104854176
11000 0.048055218009690724
12000 0.04495423153411538
13000 0.042142727748666654
14000 0.03954806820952735
15000 0.03933888172794199
16000 0.03508197650499582
17000 0.03307674442673935
18000 0.031110157317792337
19000 0.029213978713387868


# Task 2.6

In [38]:
loss_list = []
for i in range(epoch):
    model.zero_grad()
    grad_loss = (calc_loss())
    loss_list.append(grad_loss.item())
    grad_loss.backward()
    optimizer.step()
    if i % 1000 == 0:
        print(i, loss_list[-1])

0 0.025659811695655673
1000 0.025454550513251066
2000 0.025406166131561853
3000 0.02523798262047556
4000 0.026297319051073076
5000 0.025121943357162915
6000 0.025081071183934975
7000 0.02504631407411135
8000 0.025024868924132235
9000 0.02810343765793683
10000 0.025085267973898765
11000 0.024944133268340368
12000 0.024923381789746717
13000 0.024904024861548482
14000 0.024885820569112264
15000 0.02486854749121422
16000 0.03272885191572644
17000 0.024890389166389987
18000 0.059023528808741406
19000 0.04104470241768845


In [22]:
print(loss_list)

[24159.94770619082, 24337.955785971735, 24516.500455050555, 24695.497994478734, 24874.860265583477, 25054.494655895764, 25234.304026091115, 25414.18665715753, 25594.036196996836, 25773.741605679657, 25953.18709860988, 26132.25208691589, 26310.811114469943, 26488.733791049493, 26665.884721291844, 26842.123429259467, 27017.304278627485, 27191.27638872744, 27363.883546935234, 27534.964118173993, 27704.35095261886, 27871.871293037846, 28037.346683585594, 28200.592882281726, 28361.419779856937, 28519.63132813617, 28675.02548164651, 28827.394156692266, 28976.523212719407, 29122.192461399016, 29264.17570948209, 29402.24084210791, 29536.14995387036, 29665.659535543786, 29790.520724917296, 29910.47963065795, 30025.27773848096, 30134.652409111717, 30238.337477529632, 30336.063962738623, 30427.56089675255, 30512.556280560388, 30590.778173483483, 30661.955920501947, 30725.82151976286, 30782.111129558023, 30830.566710564406, 30870.93779509448, 30902.983370563714, 30926.473859442794, 30941.193172771

In [24]:
print(loss_list)

[3405.3586695216804, 3338.9877251285466, 3273.2877525798717, 3208.270013814712, 3143.945404476345, 3080.324422590061, 3017.417138961546, 2955.233169507291, 2893.7816497045924, 2833.0712113226414, 2773.1099615681824, 2713.905464750115, 2655.4647265376298, 2597.7941808567307, 2540.899679440666, 2484.7864840218244, 2429.4592611257654, 2374.9220794035455, 2321.178409416049, 2268.231125764054, 2216.082511440639, 2164.7342642681497, 2114.1875052704386, 2064.4427888223354, 2015.500114412416, 1967.3589398517138, 1920.0181957601217, 1873.4763011634268, 1827.731180037268, 1782.7802786391192, 1738.6205834760276, 1695.2486397632606, 1652.6605702378233, 1610.85209420007, 1569.8185466664095, 1529.5548975264194, 1490.0557706077186, 1451.3154625624272, 1413.3279614987805, 1376.0869652914405, 1339.5858995132164, 1303.817934939754, 1268.7760045871742, 1234.4528202502036, 1200.8408885155845, 1167.9325262320178, 1135.7198754237397, 1104.1949176402031, 1073.349487738921, 1043.1752871027459, 1013.6638962963